# Face Attribute Editing — Benchmark & Fine-Tuning Pipeline

This notebook provides a clean, modular pipeline for fine-tuning and benchmarking **3 diffusion backbones** for the task of **face attribute editing**:

1. **SD 1.5** — Stable Diffusion 1.5 baseline.
2. **SDXL** — Stable Diffusion XL base 1.0.
3. **Playground v2.5** — 1024 aesthetic (SDXL-compatible).

We target three attribute editing tasks:
- `add_eyeglasses`: Add eyeglasses.
- `make_smiling`: Make expression smiling.
- `make_older`: Make face look older.

All underlying logic is imported from the structured `src/` modules.

## 1. Environment Setup

In [ ]:
# If running in Colab/Kaggle, install required packages
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = Path("/kaggle/working").exists()

if IS_COLAB or IS_KAGGLE:
    print("Installing dependencies...")
    # Clone repo or install in editable mode if structure exists
    # !pip install -r requirements.txt
    # !pip install -e .
else:
    print("Running locally. Ensure setup.py or requirements.txt are installed.")

In [ ]:
# Add src/ package directory to sys.path
import os
project_root = Path(os.getcwd()).resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
print("Project Root set to:", project_root)

## 2. Global Configuration

In [ ]:
from src.config import CFG, PATHS, SEED, seed_everything

# Set seeds for reproducibility
seed_everything(SEED)

print("Global Config loaded successfully.")
print("Seed:", SEED)
print("Project Paths:")
for k, v in PATHS.items():
    print(f"  {k}: {v}")

## 3. Data Download & Bootstrapping

In [ ]:
from src.data.download import download_kaggle_dataset

# Set Kaggle Credentials if needed via env variables or file
# os.environ["KAGGLE_USERNAME"] = "your_username"
# os.environ["KAGGLE_KEY"] = "your_key"

print("Checking/Downloading raw CelebA & CelebAMask-HQ...")
# Uncomment to download when running end-to-end
# download_kaggle_dataset()

## 4. Preprocessing & Manifest Splitting

In [ ]:
from src.data.preprocess import run_preprocessing
from src.data.manifest import build_manifests

SMOKE_TEST = True # Set to False for full runs
max_images = 200 if SMOKE_TEST else 2000

print("Running preprocessing...")
records = run_preprocessing(max_images=max_images, force=False)

print("Building manifests...")
splits = build_manifests(records, seed=SEED, force=False)

## 5. Train Face Attribute Classifier

The classifier is an EfficientNet-B0 trained to recognize the three target attributes on cropped faces. It's used both during editing candidate selection and during benchmark evaluation.

In [ ]:
from src.models.classifier import train_classifier

print("Training or loading face attribute classifier...")
classifier_ckpt = train_classifier(smoke=SMOKE_TEST, force=False)
print("Classifier checkpoint path:", classifier_ckpt)

## 6. Train LoRA Adapters (Diffusion Models)

We fine-tune Stable Diffusion 1.5, Stable Diffusion XL, or Playground v2.5 using text-to-image scripts with standard image folder setups.

In [ ]:
from src.models.lora_training import build_diffusers_imagefolder, run_lora_training

RUN_LORA_TRAINING = False # Set to True to start training
model_id = "sd15" # Options: 'sd15', 'sdxl', 'playground25'

if RUN_LORA_TRAINING:
    split_name = "train_smoke" if SMOKE_TEST else "train"
    
    print(f"Building image folder for {model_id}...")
    data_dir = build_diffusers_imagefolder(model_id, split_name=split_name, force=False)
    
    print(f"Running LoRA training for {model_id}...")
    lora_dir = run_lora_training(model_id, data_dir, smoke=SMOKE_TEST, force=False)
    print("LoRA training directory:", lora_dir)
else:
    print("LoRA training step skipped (already trained or RUN_LORA_TRAINING=False).")

## 7. Batch Inference / Face Editing

We run inpainting with soft-mask blending for the test set, generating multiple candidates and selecting the best one.

In [ ]:
from src.inference.batch_edit import batch_edit_model

model_id = "sd15"
split_name = "test_smoke" if SMOKE_TEST else "test"
samples_per_task = 2 if SMOKE_TEST else 50

print(f"Running batch edits with model {model_id}...")
edited_meta = batch_edit_model(
    model_id=model_id,
    split_name=split_name,
    samples_per_task=samples_per_task
)
print(f"Batch editing done. Edited {len(edited_meta)} images.")

## 8. Evaluation & Metrics

In [ ]:
from src.evaluation.evaluate import evaluate_edits

print("Evaluating edited outputs...")
results_long, results_summary = evaluate_edits()
if not results_summary.empty:
    display(results_summary)
else:
    print("No results to display.")

## 9. Cross-model Benchmarking & Plotting

In [ ]:
import pandas as pd
from src.evaluation.benchmark import run_benchmark
from src.config import MODELS

# Determine if active run results_summary exists
active_csv = PATHS["run_dir"] / "metrics" / "results_summary.csv"
exports_csv = PATHS["exports_dir"] / "results_summary.csv"
csv_path = active_csv if active_csv.exists() else (exports_csv if exports_csv.exists() else None)

results_csvs = {}
if csv_path:
    print(f"Found results summary at: {csv_path}")
    for m in MODELS:
        results_csvs[m] = csv_path

print("Running cross-model benchmarking comparison...")
benchmark_res = run_benchmark(results_csvs=results_csvs if results_csvs else None)
print("Benchmark completed.")

## 10. Qualitative Visualizations & Report

In [ ]:
from src.visualization.report import generate_report
from src.visualization.qualitative_grid import make_comparison_grid

# Load long results if available
active_long = PATHS["run_dir"] / "metrics" / "results_long.csv"
exports_long = PATHS["exports_dir"] / "results_long.csv"
long_path = active_long if active_long.exists() else (exports_long if exports_long.exists() else None)

qual_grid_paths = []
if long_path and long_path.exists():
    df_long = pd.read_csv(long_path)
    # Generate side-by-side grids for tasks
    from src.config import TASKS
    for task in TASKS:
        grid_path = PATHS["run_dir"] / "reports" / f"grid_{task}.png"
        make_comparison_grid(df_long, task, grid_path)
        qual_grid_paths.append(grid_path)

print("Generating Markdown benchmark report...")
# Set results_summary to None if it doesn't exist yet
try:
    res_summary = results_summary if not results_summary.empty else None
except NameError:
    res_summary = None

report_path = generate_report(
    results_summary=res_summary,
    qual_grid_paths=qual_grid_paths,
    smoke=SMOKE_TEST
)
print("Report saved to:", report_path)